# **CÀI LẠI MÔI TRƯỜNG**

In [ ]:
#Kiểm tra GPU

import sys
import platform
import subprocess

print("Hệ điều hành:", platform.platform())
print("Python:", sys.version)
print("\nThông tin GPU:")

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout or result.stderr)


In [ ]:
%%bash
#Tạo môi trường Python 3.8


set -euo pipefail

mkdir -p /content/hr17_tools

if [ ! -x /content/hr17_tools/bin/micromamba ]; then
    curl -fLsS \
      https://micro.mamba.pm/api/micromamba/linux-64/latest \
      -o /content/hr17_tools/micromamba.tar.bz2

    tar -xjf /content/hr17_tools/micromamba.tar.bz2 \
      -C /content/hr17_tools bin/micromamba
fi

export MAMBA_ROOT_PREFIX=/content/hr17_mamba

if [ ! -x /content/hr17_env/bin/python ]; then
    /content/hr17_tools/bin/micromamba create -y \
      -p /content/hr17_env \
      --override-channels -c conda-forge \
      python=3.8 pip
fi

/content/hr17_env/bin/python --version


In [ ]:
%%bash
#Cài PyTorch và NumPy
#Cài PyTorch theo phiên bản trong log cũ

set -euo pipefail

/content/hr17_env/bin/python -m pip install \
  "pip==24.2" "numpy==1.23.0"

/content/hr17_env/bin/python -m pip install \
  torch==2.4.1 torchvision==0.19.1 \
  --index-url https://download.pytorch.org/whl/cu121


In [ ]:
%%bash
#Kiểm tra tính toán PyTorch trên GPU

set -euo pipefail

/content/hr17_env/bin/python - <<'PY'
import sys
import numpy as np
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA của PyTorch:", torch.version.cuda)
print("Nhận GPU:", torch.cuda.is_available())

assert torch.cuda.is_available(), "PyTorch chưa nhận GPU"

print("Tên GPU:", torch.cuda.get_device_name(0))

x = torch.ones((100, 100), device="cuda")
y = x @ x
torch.cuda.synchronize()

print("Kết quả kiểm tra:", y[0, 0].item())
print("ĐÃ KIỂM TRA PYTORCH TRÊN GPU THÀNH CÔNG")
PY


In [ ]:
#Tải Isaac Gym và ba ZIP đã lưu
from google.colab import files
from pathlib import Path
import re

names = ['IsaacGym_Preview_4_Package.tar', '20261118_212729_418901.zip',
         '20261118_215810_217119.zip', '20261118_221112_587011.zip']
for name in names:
    target = Path('/content') / name
    if target.is_file():
        print('Đã có:', target)
        continue
    print('Hãy chọn:', name)
    uploaded = files.upload()
    matches = [key for key in uploaded
               if re.sub(r' \(\d+\)(?=\.[^.]+$)', '', Path(key).name) == name]
    assert len(matches) == 1, 'Chọn đúng file: ' + name
    target.write_bytes(uploaded[matches[0]])
    print('Đã lưu:', target, '|', target.stat().st_size, 'bytes')


In [ ]:
#Kiểm tra và giải nén Isaac Gym
from pathlib import Path
import tarfile

archive = Path("/content/IsaacGym_Preview_4_Package.tar")

assert archive.exists(), "Chưa thấy file. Hãy kiểm tra tải lên đã hoàn tất."

print("Dung lượng:", archive.stat().st_size, "bytes")
assert archive.stat().st_size == 201141673, (
    "Dung lượng khác file đã kiểm tra trên máy. Hãy kiểm tra lại."
)

with tarfile.open(archive, "r:*") as package:
    package.extractall("/content", filter="data")

assert Path("/content/isaacgym/python/setup.py").exists()
assert Path(
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64/gym_38.so"
).exists()

print("ĐÃ GIẢI NÉN ISAAC GYM PREVIEW 4 THÀNH CÔNG")


In [ ]:
%%bash
#Tải repo đúng commit và tìm URDF H1

set -euo pipefail

if [ ! -d /content/hr17_repo ]; then
    git clone https://github.com/LeCAR-Lab/human2humanoid.git \
      /content/hr17_repo
fi

git -C /content/hr17_repo checkout --detach 750f1fa

echo "Commit đang dùng:"
git -C /content/hr17_repo rev-parse --short HEAD

test -f /content/hr17_repo/legged_gym/resources/robots/h1/urdf/h1.urdf

echo "ĐÃ TÌM THẤY MÔ HÌNH H1"


In [ ]:
%%writefile /content/hr17_restore_latest_session.py
#Ghi script khôi phục phiên mới
ARCHIVE_HASHES = {'20261118_212729_418901.zip': '702e04e7fdba317adf7cd2be6a1863baa3971e32bf3e92991f73cbd10b448a00', '20261118_215810_217119.zip': 'd170dfaf78cc0fb9beb12436a902793aabd3f5c187244dd35556cb7aa98d7448', '20261118_221112_587011.zip': 'ab86848d750f679aa672d63b40709e0512e622ccaeef8190813a1a9e4fef3589'}
"""Restore verified saved experiments and their environment; never train or evaluate."""
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import stat
import subprocess
import sys
import zipfile

CONTENT = Path('/content')
PYTHON = CONTENT / 'hr17_env/bin/python'

def sha(data): return hashlib.sha256(data).hexdigest()

def command(args, cwd=None, capture=False):
    result = subprocess.run(args, cwd=cwd, check=True, text=True,
                            stdout=subprocess.PIPE if capture else None,
                            stderr=subprocess.STDOUT if capture else None)
    return result.stdout if capture else None

def write_same_or_new(destination, data):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.is_file() and destination.read_bytes() == data, 'Existing file differs: ' + str(destination)
    else:
        destination.write_bytes(data)

def verified_zip(name):
    path = CONTENT / name
    assert path.is_file(), 'Upload: ' + name
    assert sha(path.read_bytes()) == ARCHIVE_HASHES[name], 'ZIP differs from verified original: ' + name
    z = zipfile.ZipFile(path)
    assert z.testzip() is None
    manifest = json.loads(z.read('SHA256_manifest.json'))
    for filename, digest in manifest.items():
        assert sha(z.read(filename)) == digest, filename
    return z

def unpack_member(z, name, root, relative=None):
    member = z.getinfo(name)
    assert not stat.S_ISLNK(member.external_attr >> 16), 'Symlink in ZIP'
    relative = name if relative is None else relative
    target = (root / relative).resolve()
    assert os.path.commonpath([str(root.resolve()), str(target)]) == str(root.resolve())
    if not member.is_dir(): write_same_or_new(target, z.read(name))

def ensure_git(path, url, commit, patch):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists(): command(['git', 'clone', url, str(path)])
    assert (path / '.git').exists(), 'Expected git repository: ' + str(path)
    current = command(['git', 'rev-parse', 'HEAD'], cwd=str(path), capture=True).strip()
    if current != commit:
        assert not command(['git', 'status', '--porcelain'], cwd=str(path), capture=True).strip(), 'Existing repository has changes'
        command(['git', 'fetch', 'origin', commit], cwd=str(path))
        command(['git', 'checkout', '--detach', commit], cwd=str(path))
    current_diff = command(['git', 'diff', '--no-ext-diff'], cwd=str(path), capture=True)
    if current_diff.splitlines() != patch.splitlines():
        assert not current_diff.strip(), 'Existing repository patch differs: ' + str(path)
        if patch.strip():
            patch_file = CONTENT / ('hr17_restore_' + path.name + '.patch')
            patch_file.write_text(patch, encoding='utf-8')
            command(['git', 'apply', '--check', str(patch_file)], cwd=str(path))
            command(['git', 'apply', str(patch_file)], cwd=str(path))
    assert command(['git', 'rev-parse', 'HEAD'], cwd=str(path), capture=True).strip() == commit
    assert command(['git', 'diff', '--no-ext-diff'], cwd=str(path), capture=True).splitlines() == patch.splitlines()

def main():
    assert sys.version_info[:2] == (3, 8), 'Run with /content/hr17_env/bin/python'
    assert Path('/content/isaacgym/python/setup.py').is_file(), 'Extract Isaac Gym first'
    # Verify all input files before installing or restoring anything.
    packages = {name: verified_zip(name) for name in ARCHIVE_HASHES}
    training = packages['20261118_212729_418901.zip']
    training_status = json.loads(training.read('audit/experiment_status.json'))
    assert training_status['status'] == 'completed'
    baseline = 'runs/H2O_seed1_20261116_005911/'
    metadata = json.loads(training.read(baseline + 'run_metadata.json'))
    ensure_git(CONTENT / 'hr17_repo', 'https://github.com/LeCAR-Lab/human2humanoid.git',
               metadata['repo_commit'], training.read(baseline + 'repo_changes.patch').decode())
    for name in ['SMPLSim', 'smplx']:
        ensure_git(CONTENT / 'hr17_deps' / name, 'https://github.com/ZhengyiLuo/' + name + '.git',
                   training.read(baseline + name + '_commit.txt').decode().strip(),
                   training.read(baseline + name + '_changes.patch').decode())
    freeze = training.read(baseline + 'environment_freeze.txt').decode()
    pinned = [line.strip() for line in freeze.splitlines()
              if re.fullmatch(r'[A-Za-z0-9_.-]+==[^\s]+', line.strip())]
    requirements = CONTENT / 'hr17_saved_requirements.txt'
    requirements.write_text('\n'.join(pinned) + '\n')
    print('RESTORING SAVED PACKAGE VERSIONS:', len(pinned), flush=True)
    command([str(PYTHON), '-m', 'pip', 'install', '--no-deps',
             '--extra-index-url', 'https://download.pytorch.org/whl/cu121', '-r', str(requirements)])
    editable = [CONTENT / 'isaacgym/python', CONTENT / 'hr17_repo/rsl_rl',
                CONTENT / 'hr17_repo/legged_gym', CONTENT / 'hr17_repo/phc',
                CONTENT / 'hr17_deps/SMPLSim', CONTENT / 'hr17_deps/smplx']
    command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '--no-build-isolation']
            + [part for directory in editable for part in ['-e', str(directory)]])
    actual = command([str(PYTHON), '-m', 'pip', 'freeze'], capture=True)
    lines = lambda s: {line.strip() for line in s.splitlines() if line.strip() and not line.startswith('#')}
    missing, extra = sorted(lines(freeze)-lines(actual)), sorted(lines(actual)-lines(freeze))
    (CONTENT / 'hr17_restore_environment_audit.json').write_text(json.dumps(dict(
        matches_saved_freeze=not missing and not extra, missing_or_different=missing, extra_or_different=extra), indent=2))
    assert not missing and not extra, 'Environment differs; see /content/hr17_restore_environment_audit.json: ' + str((missing, extra))
    group = CONTENT / 'hr17_three_seeds/20261118_212729_418901'
    for name in training.namelist():
        if name.startswith('runs/'):
            unpack_member(training, name, CONTENT / 'hr17_training', name[len('runs/'):])
        elif name.startswith('audit/'):
            unpack_member(training, name, group, name[len('audit/'):])
    for seed, path in training_status['runs'].items():
        folder = Path(path)
        assert folder.parent == CONTENT / 'hr17_training'
        meta = json.loads((folder / 'run_metadata.json').read_text())
        assert meta['seed'] == int(seed) and meta['status'] == 'completed'
        assert sha((folder / 'model_200.pt').read_bytes()) == meta['final_checkpoint_sha256']
    # Pointers require the ZIP alongside its extracted directory.
    shutil.copy2(CONTENT / '20261118_212729_418901.zip', group.with_suffix('.zip'))
    (group.parent / 'latest_archive.txt').write_text(str(group.with_suffix('.zip')))
    for filename, root in [('20261118_215810_217119.zip', 'hr17_three_seed_evaluation'),
                           ('20261118_221112_587011.zip', 'hr17_reward_diagnosis')]:
        z = packages[filename]
        folder = CONTENT / root / Path(filename).stem
        assert json.loads(z.read('experiment_status.json'))['status'] == 'completed'
        for name in z.namelist(): unpack_member(z, name, folder)
        shutil.copy2(CONTENT / filename, folder.with_suffix('.zip'))
        (folder.parent / 'latest_archive.txt').write_text(str(folder.with_suffix('.zip')))
    for filename, archive in [('hr17_evaluate_three_seeds.py', '20261118_215810_217119.zip'),
                               ('hr17_diagnose_rewards.py', '20261118_221112_587011.zip')]:
        write_same_or_new(CONTENT / filename, packages[archive].read(filename))
    baseline_dir = CONTENT / 'hr17_training/H2O_seed1_20261116_005911'
    (baseline_dir.parent / 'latest_run.txt').write_text(str(baseline_dir))
    config_dir = CONTENT / 'hr17_checkpoint_comparison_input'
    for name in ['env_cfg.json', 'train_cfg.json']:
        write_same_or_new(config_dir / name, (baseline_dir / ('source_' + name)).read_bytes())
    env = os.environ.copy()
    env['PATH'] = '/content/hr17_env/bin:' + env.get('PATH', '')
    # Match the earlier successful H2O import check. Do not globally inject
    # bundled Isaac Gym libraries into the dynamic loader search path.
    runtime = Path('/content/hr17_runtime_environment.json')
    saved_runtime = json.loads(runtime.read_text()) if runtime.is_file() else {}
    env['LD_LIBRARY_PATH'] = saved_runtime.get('LD_LIBRARY_PATH') if saved_runtime.get('native_h2o_import_verified') else (
        '/content/hr17_env/lib:' + env.get('LD_LIBRARY_PATH', ''))
    env['PYTHONFAULTHANDLER'] = '1'
    env['MAX_JOBS'] = '2'
    check = '''print("CHECK 1: importing Isaac Gym", flush=True)
import isaacgym
print("CHECK 2: importing PyTorch", flush=True)
import torch, numpy as np
print("CHECK 3: importing H2O", flush=True)
from legged_gym.envs import LeggedRobot
assert torch.__version__ == "2.4.1+cu121" and np.__version__ == "1.23.0"
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))
print("IMPORT MOI TRUONG H2O THANH CONG")
'''
    subprocess.run([str(PYTHON), '-u', '-c', check], cwd='/content/hr17_repo', env=env, check=True)
    runtime.write_text(json.dumps({
        'LD_LIBRARY_PATH': env['LD_LIBRARY_PATH'],
        'native_h2o_import_verified': True,
    }, indent=2), encoding='utf-8')
    for z in packages.values(): z.close()
    print('KHOI PHUC PHIEN MOI THANH CONG — 3 SEED + DANH GIA + CHAN DOAN', flush=True)
    print('NO TRAINING OR EVALUATION WAS RUN', flush=True)

if __name__ == '__main__': main()


In [ ]:
#Thực thi khôi phục thư viện, checkpoint và dữ liệu
from pathlib import Path
import os
import subprocess

env = os.environ.copy()
env['PATH'] = '/content/hr17_env/bin:' + env.get('PATH', '')
env['LD_LIBRARY_PATH'] = '/content/hr17_env/lib:' + env.get('LD_LIBRARY_PATH', '')
env['PYTHONUNBUFFERED'] = '1'
with Path('/content/hr17_restore_console.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        ['/content/hr17_env/bin/python', '-u', '/content/hr17_restore_latest_session.py'],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    return_code = process.wait()
print('Mã kết thúc:', return_code)
assert return_code == 0, 'Gửi output lỗi; log: /content/hr17_restore_console.log'


In [ ]:
from pathlib import Path
import json

log = Path("/content/hr17_restore_console.log")
if log.is_file():
    print("=== 80 DÒNG CUỐI CỦA LOG ===")
    print("\n".join(
        log.read_text(errors="replace").splitlines()[-80:]
    ))

audit = Path("/content/hr17_restore_environment_audit.json")
if audit.is_file():
    data = json.loads(audit.read_text())
    print("\nThư viện khớp bản đã lưu:",
          data.get("matches_saved_freeze"))

checkpoint = Path(
    "/content/hr17_training/"
    "H2O_seed1_20261116_005911/model_50.pt"
)
print("Checkpoint model_50 đã khôi phục:", checkpoint.is_file())

In [ ]:
from google.colab import files
from pathlib import Path
import re
import runpy

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_recheck_native_import(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Chọn file hr17_recheck_native_import.py"

script = Path("/content/hr17_recheck_native_import.py")
script.write_bytes(uploaded[matches[0]])

runpy.run_path(str(script), run_name="__main__")

In [ ]:
#Xác nhận ba seed và dữ liệu đã sẵn sàng
from pathlib import Path
import json
root = Path('/content/hr17_three_seeds')
group = Path((root / 'latest_archive.txt').read_text().strip()).with_suffix('')
status = json.loads((group / 'experiment_status.json').read_text())
for seed, run in status['runs'].items():
    folder = Path(run)
    for filename in ['model_initial.pt', 'model_50.pt', 'model_200.pt', 'env_cfg.json', 'train_cfg.json']:
        assert (folder / filename).is_file(), str(folder / filename)
    print('Đã khôi phục seed', seed, ':', folder)
for name in ['hr17_three_seed_evaluation', 'hr17_reward_diagnosis']:
    folder = Path((Path('/content') / name / 'latest_archive.txt').read_text().strip()).with_suffix('')
    assert json.loads((folder / 'experiment_status.json').read_text())['status'] == 'completed'
    print('Đã khôi phục:', name)
print('KHÔI PHỤC THÀNH CÔNG — dừng tại đây để chuẩn bị phép thử mới; không cần huấn luyện lại')


# **Kiểm tra giữ mục tiêu hai khuỷu tay ở bước đầu rồi trả quyền cho policy có giúp robot duy trì lâu hơn không?**

In [ ]:
#Tải Script
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_test_elbow_episode(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Chọn file hr17_test_elbow_episode.py"

script = Path("/content/hr17_test_elbow_episode.py")
script.write_bytes(uploaded[matches[0]])

print("ĐÃ LƯU:", script)

In [ ]:
from google.colab import files
from pathlib import Path
import re
import runpy

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_native_diagnosis_v3(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Chọn file hr17_native_diagnosis_v3.py"

script = Path("/content/hr17_native_diagnosis_v3.py")
script.write_bytes(uploaded[matches[0]])

runpy.run_path(str(script), run_name="__main__")

In [ ]:
#Chạy đối chứng trên toàn episode
#chạy 12 rollout: ba seed × hai checkpoint × hai cách điều khiển
from pathlib import Path
import json
import os
import subprocess

script = Path("/content/hr17_test_elbow_episode.py")
assert script.is_file(), "Hãy chạy ô tải script trước."

runtime = json.loads(
    Path("/content/hr17_runtime_environment.json").read_text()
)
assert runtime.get("native_h2o_import_verified"), (
    "Cần hoàn thành kiểm tra native import trước."
)

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = runtime["LD_LIBRARY_PATH"]
env.update(
    MAX_JOBS="2",
    MPLBACKEND="Agg",
    PYTHONUNBUFFERED="1",
    PYTHONFAULTHANDLER="1",
)

log_path = Path("/content/hr17_elbow_episode_console.log")

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        ["/content/hr17_env/bin/python", "-u", str(script)],
        cwd="/content/hr17_repo",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)
assert return_code == 0, (
    "Phép thử chưa hoàn tất. Chạy ô tải ZIP bên dưới để lấy log."
)

**Đã xác minh** 292 tệp, đủ 12 rollout; checkpoint khớp dữ liệu huấn luyện, trạng thái đầu giữa từng cặp trùng nhau và chỉ hai action khuỷu tay được thay trong bước đầu.

| Seed / checkpoint | Reset lệnh gốc | Reset sau can thiệp |
|---|---:|---:|
| Seed 1 — khởi tạo | 1,34 s | 1,30 s |
| Seed 1 — cuối | 0,26 s | 0,26 s |
| Seed 2 — khởi tạo | 1,72 s | 1,60 s |
| Seed 2 — cuối | 0,12 s | 0,12 s |
| Seed 3 — khởi tạo | 1,24 s | 1,24 s |
| Seed 3 — cuối | 0,20 s | 0,20 s |

**Có ba phát hiện chính:**
- Sai số `global MPJPE` trong 0,10 giây đầu giảm `8,89–18,88%` ở cả sáu cặp. Đây chỉ là cửa sổ đầu ngắn.

- Ngay bước điều khiển thứ hai, cả 12 khuỷu tay trong các rollout can thiệp lại bão hòa mô-men; vận tốc trở lại khoảng `20 rad/s`. Tác dụng giảm xung ban đầu không được duy trì khi trả quyền cho policy.

- Không cặp nào kéo dài episode: bốn cặp không đổi, hai cặp kết thúc sớm hơn. Cả 12 rollout đều reset do `gravity_x`, trước khi hết motion `5,233 giây`.

In [ ]:
#Tải kết quả về máy
from google.colab import files
from pathlib import Path
import json

pointer = Path("/content/hr17_elbow_episode/latest_archive.txt")
assert pointer.is_file(), (
    "Chưa có gói kết quả; hãy gửi output ô chạy đối chứng."
)

archive = Path(pointer.read_text().strip())
assert archive.is_file(), str(archive)

status_file = archive.with_suffix("") / "experiment_status.json"
if status_file.is_file():
    status = json.loads(status_file.read_text())
    print("Trạng thái:", status["status"])

print("Tải gói:", archive.name)
files.download(str(archive))

# **Chuẩn bị 6 chuyển động ID/OOD — không cần GPU**

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import io
import json
import re
import zipfile

# Chọn file HR17_Motion_ID_OOD_6clip.zip trong thư mục ML
uploaded = files.upload()
assert len(uploaded) == 1, "Chỉ chọn một file ZIP."

name, raw = next(iter(uploaded.items()))
assert name.lower().endswith(".zip"), "Hãy chọn file ZIP."

with zipfile.ZipFile(io.BytesIO(raw)) as package:
    assert package.testzip() is None, "File ZIP bị lỗi."
    manifest = json.loads(package.read("manifest.json"))

    motions = manifest["motions"]
    expected_ids = {"id01", "id02", "id03", "ood01", "ood02", "ood03"}
    assert len(motions) == 6
    assert {row["id"] for row in motions} == expected_ids

    # Kiểm tra toàn bộ dữ liệu trước khi ghi ra thư mục
    checked = {}
    for row in motions:
        filename = row["file"]
        assert re.fullmatch(r"(?:id|ood)0[1-3]\.pkl", filename)
        payload = package.read(filename)
        actual_hash = hashlib.sha256(payload).hexdigest()
        assert actual_hash == row["sha256"], f"File không khớp: {filename}"
        checked[filename] = payload

    output = Path("/content/hr17_motion_id_ood")
    output.mkdir(parents=True, exist_ok=True)

    for filename, payload in checked.items():
        (output / filename).write_bytes(payload)

    (output / "manifest.json").write_bytes(
        package.read("manifest.json")
    )
    (output / "HUONG_DAN.txt").write_bytes(
        package.read("HUONG_DAN.txt")
    )

(output / "HR17_Motion_ID_OOD_6clip.zip").write_bytes(raw)

print("\nĐÃ KIỂM TRA VÀ CHUẨN BỊ ĐỦ 6 CHUYỂN ĐỘNG")
for row in motions:
    group = "Bộ huấn luyện" if row["group"] == "training_pool" else "Chưa dùng huấn luyện"
    print(f'{row["id"]} | {group} | {row["duration_s"]:.2f} giây')

print("\nThư mục:", output)
print("Mới kiểm tra dữ liệu; chưa chạy mô phỏng hoặc đánh giá.")

# **ID/OOD — Tải và kiểm tra script, không cần GPU**

In [ ]:
from google.colab import files
from pathlib import Path
import re
import subprocess
import sys

uploaded = files.upload()
assert len(uploaded) == 1, "Chỉ chọn một file script."

name, payload = next(iter(uploaded.items()))
assert re.fullmatch(
    r"hr17_evaluate_motion_id_ood(?: \(\d+\))?\.py",
    name
), "Hãy chọn file hr17_evaluate_motion_id_ood.py"

script = Path("/content/hr17_evaluate_motion_id_ood.py")
script.write_bytes(payload)

assert Path("/content/hr17_motion_id_ood/manifest.json").is_file(), (
    "Chưa có dữ liệu. Hãy chạy lại ô tải ZIP sáu chuyển động."
)

# Chế độ mặc định chỉ kiểm tra dữ liệu và script.
# Không import Isaac Gym, không sử dụng GPU.
subprocess.run(
    [sys.executable, str(script)],
    check=True
)

In [ ]:
from pathlib import Path

script = Path("/content/hr17_evaluate_motion_id_ood.py")
assert "def normalized_motion_keys" in script.read_text(), (
    "Bạn đang dùng bản cũ. Hãy tải lại script vừa sửa."
)
print("ĐÃ NHẬN SCRIPT SỬA LỖI TÊN MOTION.")

In [ ]:
#kiểm tra dữ liệu và script, không cần GPU
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "/content/hr17_evaluate_motion_id_ood.py"],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "Kiểm tra chưa thành công."

In [ ]:
#Kiểm tra trước khi đánh giá ID/OOD
from pathlib import Path
import hashlib
import json

run = Path("/content/hr17_training/H2O_seed1_20261116_005911")

required = [
    Path("/content/hr17_env/bin/python"),
    Path("/content/hr17_repo/legged_gym/legged_gym/envs/base/legged_robot.py"),
    run / "model_50.pt",
    run / "env_cfg.json",
    run / "train_cfg.json",
    run / "run_metadata.json",
    Path("/content/hr17_runtime_environment.json"),
    Path("/content/hr17_motion_id_ood/manifest.json"),
    Path("/content/hr17_evaluate_motion_id_ood.py"),
]

missing = [p for p in required if not p.is_file()]

if missing:
    print("CẦN KHÔI PHỤC CÁC THÀNH PHẦN:")
    for p in missing:
        print("-", p)
else:
    runtime = json.loads(
        Path("/content/hr17_runtime_environment.json").read_text()
    )
    checkpoint_hash = hashlib.sha256(
        (run / "model_50.pt").read_bytes()
    ).hexdigest()

    assert checkpoint_hash == (
        "59c0bc4d25d8a3d5b468016ad1081fdfc2c44f41899ad4b530bc9573513d28fc"
    ), "Checkpoint không khớp bản đã chọn."

    if runtime.get("native_h2o_import_verified"):
        print("ĐỦ FILE VÀ CHECKPOINT KHỚP.")
        print("Có bản ghi kiểm tra native H2O; bước tiếp theo kiểm tra chạy GPU.")
    else:
        print("ĐỦ FILE, NHƯNG CẦN KIỂM TRA NATIVE H2O.")

In [ ]:
#Chạy đánh giá 6 chuyển động ID/OOD
from pathlib import Path
import json
import os
import subprocess

python = Path("/content/hr17_env/bin/python")
script = Path("/content/hr17_evaluate_motion_id_ood.py")
runtime_file = Path("/content/hr17_runtime_environment.json")

assert python.is_file(), "Thiếu môi trường Python 3.8."
assert script.is_file(), "Hãy chạy ô 17 để tải script."
assert runtime_file.is_file(), "Thiếu bản ghi kiểm tra native."

runtime = json.loads(runtime_file.read_text())
assert runtime.get("native_h2o_import_verified"), (
    "Kiểm tra native H2O chưa thành công."
)

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = runtime["LD_LIBRARY_PATH"]
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"
env["MAX_JOBS"] = "2"
env["MPLBACKEND"] = "Agg"

log_path = Path("/content/hr17_motion_evaluation_console.log")

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [str(python), "-u", str(script), "--run"],
        cwd="/content/hr17_repo",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)
assert return_code == 0, "Gửi output lỗi và dòng RESULT ZIP nếu có."

| Chuyển động | Nhóm | Global MPJPE | Thời gian đến reset |
|---|---|---:|---:|
| Đấm phải | Bộ huấn luyện | 79,16 mm | 1,50 s |
| Đấm trái | Bộ huấn luyện | 92,23 mm | 1,38 s |
| Đấm thẳng trái | Bộ huấn luyện | 116,12 mm | 1,44 s |
| Vẫy tay | Giữ ngoài | 229,05 mm | 0,80 s |
| Đi bộ | Giữ ngoài | 255,59 mm | 1,00 s |
| Bước ngang trái | Giữ ngoài | 199,43 mm | 1,08 s |

In [ ]:
#Tải kết quả đánh giá 6 chuyển động ID/OOD
from google.colab import files
from pathlib import Path
import json

pointer = Path("/content/hr17_motion_evaluation/latest_archive.txt")
assert pointer.is_file(), "Chưa tìm thấy gói kết quả."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), str(archive)

status_file = archive.with_suffix("") / "experiment_status.json"
assert status_file.is_file(), "Thiếu bản ghi trạng thái."

status = json.loads(status_file.read_text())
print("Trạng thái:", status["status"])
print("Gói kết quả:", archive.name)

if status["status"] != "completed":
    print("Lần chạy chưa hoàn tất; vẫn tải ZIP để kiểm tra lỗi.")

files.download(str(archive))

In [ ]:
#KIỂM TRA TRƯỚC KHI THỬ KÉO DÀI EPISODE
from pathlib import Path
import hashlib
import json
import subprocess

run_dir = Path(
    "/content/hr17_training/H2O_seed1_20261116_005911"
)

required = [
    Path("/content/hr17_env/bin/python"),
    Path("/content/hr17_repo/legged_gym/legged_gym/envs/base/legged_robot.py"),
    run_dir / "model_50.pt",
    run_dir / "env_cfg.json",
    run_dir / "train_cfg.json",
    run_dir / "run_metadata.json",
    Path("/content/hr17_runtime_environment.json"),
    Path("/content/hr17_motion_id_ood/manifest.json"),
    Path("/content/hr17_evaluate_motion_id_ood.py"),
]

missing = [str(p) for p in required if not p.is_file()]

if missing:
    print("CẦN KHÔI PHỤC CÁC FILE:")
    for name in missing:
        print("-", name)
else:
    expected_sha = (
        "59c0bc4d25d8a3d5b468016ad1081fdfc2c44"
        "f41899ad4b530bc9573513d28fc"
    )
    checkpoint = run_dir / "model_50.pt"
    actual_sha = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    assert actual_sha == expected_sha, "Checkpoint khác bản đã đánh giá."

    runtime = json.loads(
        Path("/content/hr17_runtime_environment.json").read_text()
    )
    assert runtime.get("native_h2o_import_verified"), (
        "Chưa có xác nhận import H2O trong cấu hình khôi phục."
    )

    subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader"
        ],
        check=True
    )

    print("\nCHECKPOINT MODEL_50: KHỚP BẢN ĐÃ KIỂM CHỨNG")
    print("DỮ LIỆU VÀ SCRIPT: ĐÃ CÓ")
    print("SẴN SÀNG CHUẨN BỊ PHÉP THỬ KÉO DÀI EPISODE")
    print("Ô này chưa chạy mô phỏng, chưa thay đổi cấu hình.")

In [ ]:
#Tải script kiểm tra độ cao khởi tạo
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_test_initial_lift(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Hãy chọn đúng file hr17_test_initial_lift.py"

script = Path("/content/hr17_test_initial_lift.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT VÀ KIỂM TRA CÚ PHÁP")

In [ ]:
#Chạy đối chứng độ cao khởi tạo
import subprocess

result = subprocess.run([
    "/usr/bin/python3",
    "-u",
    "/content/hr17_test_initial_lift.py",
    "--run"
])

print("Mã kết thúc:", result.returncode)

| Độ nâng khởi tạo | Thời gian đến reset | Sai số Global MPJPE* |
|---|---:|---:|
| **4 cm — cấu hình gốc** | **1,50 giây** | **139,61 mm** |
| 2 cm | 1,40 giây | 166,32 mm |
| 6 cm | 1,44 giây | 156,11 mm |
| 4 cm — chạy lặp | 1,50 giây | 139,61 mm |

In [ ]:
from pathlib import Path
from google.colab import files

packages = list(Path("/content/hr17_initial_lift").glob("*.zip"))
assert packages, "Chưa tìm thấy gói kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải kết quả:", latest)
files.download(str(latest))

In [ ]:
#Tải script thử giảm chấn cổ chân
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_test_ankle_damping(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Chọn file hr17_test_ankle_damping.py"

script = Path("/content/hr17_test_ankle_damping.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT THÀNH CÔNG")

In [ ]:
#Chạy đối chứng Kd cổ chân
import subprocess

result = subprocess.run([
    "/usr/bin/python3",
    "-u",
    "/content/hr17_test_ankle_damping.py",
    "--run"
])

print("Mã kết thúc:", result.returncode)

| Kd hai cổ chân | Thời gian đến reset | Global MPJPE* |
|---|---:|---:|
| 2 — cấu hình gốc | 1,50 giây | 154,60 mm |
| **1** | **1,66 giây** | **117,03 mm** |
| 4 | 1,46 giây | 166,98 mm |
| 2 — chạy lặp | 1,50 giây | 154,60 mm |

In [ ]:
#Tải kết quả
from pathlib import Path
from google.colab import files

packages = list(Path("/content/hr17_ankle_damping").glob("*.zip"))
assert packages, "Chưa tìm thấy kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải:", latest)
files.download(str(latest))

In [ ]:
#Tải script đối chứng Kd trên năm chuyển động
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_validate_ankle_damping(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Chọn file hr17_validate_ankle_damping.py"

script = Path("/content/hr17_validate_ankle_damping.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT THÀNH CÔNG")

In [ ]:
#Chạy đối chứng năm chuyển động
import subprocess

result = subprocess.run([
    "/usr/bin/python3",
    "-u",
    "/content/hr17_validate_ankle_damping.py",
    "--run"
])

print("Mã kết thúc:", result.returncode)

| Chuyển động | Reset với Kd = 2 | Reset với Kd = 1 |
|---|---:|---:|
| Đấm trái `id02` | 1,38 s | **1,46 s** |
| Jab trái `id03` | 1,44 s | 1,44 s |
| Vẫy tay `ood01` | 0,80 s | **0,84 s** |
| Đi bộ `ood02` | 1,00 s | **1,04 s** |
| Bước ngang `ood03` | 1,08 s | **1,20 s** |

In [ ]:
#Chạy lại mô phỏng
from pathlib import Path
import hashlib
import json

script = Path("/content/hr17_validate_ankle_damping.py")

if script.is_file():
    digest = hashlib.sha256(script.read_bytes()).hexdigest()
    expected = (
        "e7bbd74c17a0ad1445ddf09a74d2237eb"
        "509fc4f09816b38bc6ccc210c626e19"
    )
    print("Script đúng phiên bản:", digest == expected)
    print("Dung lượng script:", script.stat().st_size, "bytes")
else:
    print("KHÔNG TÌM THẤY SCRIPT")

root = Path("/content/hr17_ankle_validation")
runs = [p for p in root.glob("*") if p.is_dir()]

if not runs:
    print("\nCHƯA TÌM THẤY THƯ MỤC KẾT QUẢ")
else:
    latest = max(runs, key=lambda p: p.stat().st_mtime)
    print("\nThư mục mới nhất:", latest)

    status_file = latest / "experiment_status.json"
    if status_file.is_file():
        status = json.loads(status_file.read_text(encoding="utf-8"))
        print("Trạng thái:", status.get("status"))
        print("Số lượt đã tổng hợp:", len(status.get("results", [])))
        print("Số cặp đã tổng hợp:", len(status.get("pairs", [])))
        print("Thống kê thời gian:", status.get("duration_counts"))
        if status.get("error"):
            print("Lỗi:", status["error"])
    else:
        print("Chưa có experiment_status.json")

    summaries = list(latest.glob("*/summary.json"))
    print("Số lượt có summary.json:", len(summaries))

    archive = latest.with_suffix(".zip")
    print("ZIP đã tạo:", archive.is_file())
    if archive.is_file():
        print("Đường dẫn ZIP:", archive)

In [ ]:
#Tải kết quả
from pathlib import Path
from google.colab import files

packages = list(Path("/content/hr17_ankle_validation").glob("*.zip"))
assert packages, "Chưa tìm thấy gói kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải:", latest)
files.download(str(latest))

In [ ]:
#Tải script kiểm tra chạy lặp
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_repeat_ankle_damping(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Chọn file hr17_repeat_ankle_damping.py"

script = Path("/content/hr17_repeat_ankle_damping.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT THÀNH CÔNG")

In [ ]:
#Chạy lặp đối chứng Kd cổ chân
import subprocess

command = [
    "/usr/bin/python3",
    "-u",
    "/content/hr17_repeat_ankle_damping.py",
    "--run"
]

with open("/content/hr17_ankle_repeat_console.log", "w") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

print("Mã kết thúc:", return_code)

| Chuyển động | Kd = 2 | Kd = 1 qua ba lần chạy |
|---|---:|---:|
| Đấm phải `id01` | 1,50 giây | **1,66 giây cả ba lần** |
| Bước ngang `ood03` | 1,08 giây | **1,20–1,22 giây** |

In [ ]:
#Tải kết quả
from pathlib import Path
from google.colab import files

packages = list(Path("/content/hr17_ankle_repeat").glob("*.zip"))
assert packages, "Chưa tìm thấy gói kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải:", latest)
files.download(str(latest))

In [ ]:
#Tải script tinh chỉnh một chuyển động
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_finetune_single_motion(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Chọn file hr17_finetune_single_motion.py"

script = Path("/content/hr17_finetune_single_motion.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT THÀNH CÔNG")

In [ ]:
#Tinh chỉnh 100 cập nhật và đánh giá bốn mốc
import subprocess

command = [
    "/usr/bin/python3",
    "-u",
    "/content/hr17_finetune_single_motion.py",
    "--run"
]

with open("/content/hr17_focus_training_console.log", "w") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

print("Mã kết thúc:", return_code)

| Số cập nhật thêm | Thời gian đến reset | Global MPJPE* |
|---|---:|---:|
| **0 — trước tinh chỉnh** | **1,66 giây** | **65,73 mm** |
| 25 | 1,58 giây | 69,42 mm |
| 50 | 1,14 giây | 73,02 mm |
| 100 | 0,24 giây | 97,25 mm |

In [ ]:
#Tải kết quả để sao lưu
from pathlib import Path
from google.colab import files

packages = list(Path("/content/hr17_focus_training").glob("*.zip"))
assert packages, "Chưa tìm thấy gói kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải:", latest)
files.download(str(latest))

In [ ]:
#Tải script kiểm tra khởi đầu huấn luyện
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_test_training_start_zero(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Chọn file hr17_test_training_start_zero.py"

script = Path("/content/hr17_test_training_start_zero.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT THÀNH CÔNG")

In [ ]:
#Huấn luyện từ đầu clip và đánh giá
import subprocess

command = [
    "/usr/bin/python3",
    "-u",
    "/content/hr17_test_training_start_zero.py",
    "--run"
]

with open("/content/hr17_start_zero_console.log", "w") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

print("Mã kết thúc:", return_code)

| Cập nhật thêm | Khởi đầu ngẫu nhiên — lượt trước | Khởi đầu giây 0 — lượt mới |
|---|---:|---:|
| **0** | **1,66 giây** | **1,66 giây** |
| 25 | 1,58 giây | 1,12 giây |
| 50 | 1,14 giây | 0,26 giây |
| 100 | 0,24 giây | 0,22 giây |

In [ ]:
#Tải kết quả
from pathlib import Path
from google.colab import files

packages = list(Path("/content/hr17_focus_start_zero").glob("*.zip"))
assert packages, "Chưa tìm thấy gói kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải:", latest)
files.download(str(latest))

In [ ]:
#Tải script chẩn đoán PPO
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_diagnose_ppo_5updates(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Chọn file hr17_diagnose_ppo_5updates.py"

script = Path("/content/hr17_diagnose_ppo_5updates.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT THÀNH CÔNG")

In [ ]:
#Chạy chẩn đoán PPO — 5 cập nhật
import subprocess

command = [
    "/usr/bin/python3",
    "-u",
    "/content/hr17_diagnose_ppo_5updates.py",
    "--run"
]

with open("/content/hr17_ppo_diagnostics_console.log", "w") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

print("Mã kết thúc:", return_code)

In [ ]:
#Tải kết quả
from pathlib import Path
from google.colab import files

packages = list(Path("/content/hr17_ppo_diagnostics").glob("*.zip"))
assert packages, "Chưa tìm thấy gói kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải:", latest)
files.download(str(latest))

# **Phép đối chứng clipping chung và clipping riêng actor–critic**

In [ ]:
#Tải script đối chứng clipping PPO
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_compare_ppo_clipping(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Chọn file hr17_compare_ppo_clipping.py"

script = Path("/content/hr17_compare_ppo_clipping.py")
script.write_bytes(uploaded[matches[0]])

compile(script.read_text(encoding="utf-8"), str(script), "exec")
print("ĐÃ TẢI SCRIPT THÀNH CÔNG")

In [ ]:
#Chạy đối chứng clipping chung và riêng
import subprocess

command = [
    "/usr/bin/python3",
    "-u",
    "/content/hr17_compare_ppo_clipping.py",
    "--run"
]

with open("/content/hr17_clipping_comparison_console.log", "w") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

print("Mã kết thúc:", return_code)

| Phương án | Thời gian trước reset | Sai số MPJPE ↓ |
|---|---:|---:|
| Trước huấn luyện | 1,66 giây | 135,62 mm |
| Clipping chung — 5 cập nhật | 1,56 giây | 166,05 mm |
| Clipping riêng — 5 cập nhật | **1,74 giây** | **120,50 mm** |

In [ ]:
#Tải kết quả đối chứng clipping
from pathlib import Path
from google.colab import files

packages = list(
    Path("/content/hr17_ppo_clipping_comparison").glob("*.zip")
)
assert packages, "Chưa tìm thấy gói kết quả."

latest = max(packages, key=lambda p: p.stat().st_mtime)
print("Tải:", latest)
files.download(str(latest))

In [ ]:
from google.colab import files
from pathlib import Path
import re
import runpy

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_recheck_native_import(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_recheck_native_import.py"

script = Path("/content/hr17_recheck_native_import.py")
script.write_bytes(uploaded[matches[0]])
runpy.run_path(str(script), run_name="__main__")

In [ ]:
from google.colab import files
from pathlib import Path
import re
import runpy

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_native_diagnosis_v3(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_native_diagnosis_v3.py"

script = Path("/content/hr17_native_diagnosis_v3.py")
script.write_bytes(uploaded[matches[0]])

_ = runpy.run_path(str(script), run_name="__main__")

# **Kiểm chứng clipping riêng trên 6 chuyển động**

In [ ]:
#Tải script kiểm chứng
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_validate_clipping_6motions(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_validate_clipping_6motions.py"

script = Path("/content/hr17_validate_clipping_6motions.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA CHẠY ĐÁNH GIÁ")

In [ ]:
#Chạy kiểm chứng 6 chuyển động
import subprocess
from pathlib import Path

source = Path(
    "/content/hr17_ppo_clipping_comparison/"
    "20261124_085613_561254.zip"
)
assert source.is_file(), "Không thấy ZIP đối chứng trong phiên hiện tại."

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_validate_clipping_6motions.py",
    "--run",
    "--source-zip", str(source),
]

log_path = Path("/content/hr17_clipping_validation_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

| Chuyển động | Thời gian trước reset: gốc → clipping riêng | MPJPE: gốc → clipping riêng ↓ |
|---|---:|---:|
| id01 | 1,66 → **1,74 giây** | 160,20 → **138,96 mm** |
| id02 | 1,46 → **1,58 giây** | 175,91 → **147,17 mm** |
| id03 | 1,44 → 1,42 giây | 194,36 → 209,49 mm |
| ood01 | 0,84 → 0,82 giây | 213,43 → 220,88 mm |
| ood02 | 1,04 → 0,96 giây | 326,08 → 333,21 mm |
| ood03 | 1,21 → 1,24 giây | 354,41 → 354,88 mm |

In [ ]:
#Tải kết quả về máy
from google.colab import files
from pathlib import Path

archives = list(Path("/content/hr17_clipping_validation").glob("*.zip"))
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))

# **So sánh hướng thân robot với thân tham chiếu trên id01, id02 và ood02**

In [ ]:
#Tải script kiểm tra hướng thân
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_probe_torso_orientation(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_probe_torso_orientation.py"

script = Path("/content/hr17_probe_torso_orientation.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA CHẠY MÔ PHỎNG")

In [ ]:
#Chạy 6 lượt kiểm tra
import subprocess
from pathlib import Path

source = Path(
    "/content/hr17_ppo_clipping_comparison/"
    "20261124_085613_561254.zip"
)
assert source.is_file(), "Không thấy ZIP đối chứng trong phiên hiện tại."

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_probe_torso_orientation.py",
    "--run",
    "--source-zip", str(source),
]

log_path = Path("/content/hr17_torso_orientation_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

| Chuyển động | Hướng thân tham chiếu lúc reset | Hướng thân robot |
|---|---|---|
| id01 — đấm phải | Giữ thẳng | Nghiêng khoảng **45°** |
| id02 — đấm trái | Giữ thẳng | Nghiêng khoảng **46°** |
| ood02 — đi bộ | Nghiêng khoảng **14°** | Nghiêng khoảng **46°**, sai lệch hướng trọng lực gần **60°** |

In [ ]:
#Tải ZIP kết quả
from google.colab import files
from pathlib import Path

archives = list(Path("/content/hr17_torso_orientation").glob("*.zip"))
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))

# **Kiểm tra observation thực sự được truyền vào mạng**

In [ ]:
#Tải script kiểm tra đầu vào mạng
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_probe_actor_inputs(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_probe_actor_inputs.py"

script = Path("/content/hr17_probe_actor_inputs.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA CHẠY MÔ PHỎNG")

In [ ]:
#Chạy kiểm tra
import subprocess
from pathlib import Path

source = Path(
    "/content/hr17_ppo_clipping_comparison/"
    "20261124_085613_561254.zip"
)
assert source.is_file(), "Không thấy ZIP đối chứng trong phiên hiện tại."

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_probe_actor_inputs.py",
    "--run",
    "--source-zip", str(source),
]

log_path = Path("/content/hr17_actor_input_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

In [ ]:
#Tải kết quả
from google.colab import files
from pathlib import Path

archives = list(Path("/content/hr17_actor_input_probe").glob("*.zip"))
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))

# **Phép đối chứng chỉ hiệu chỉnh [44:47] ở lần gọi mạng đầu tiên.**

In [ ]:
#Tải script đối chứng
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_test_startup_gravity(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_test_startup_gravity.py"

script = Path("/content/hr17_test_startup_gravity.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA CHẠY ĐỐI CHỨNG")

In [ ]:
#Chạy đối chứng trọng lực đầu tiên
import subprocess
from pathlib import Path

source = Path(
    "/content/hr17_ppo_clipping_comparison/"
    "20261124_085613_561254.zip"
)
assert source.is_file(), "Không thấy ZIP nguồn trong phiên hiện tại."

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_test_startup_gravity.py",
    "--run",
    "--source-zip", str(source),
]

log_path = Path("/content/hr17_startup_gravity_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

In [ ]:
#Tải kết quả
from google.colab import files
from pathlib import Path

archives = list(Path("/content/hr17_startup_gravity_test").glob("*.zip"))
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))

In [ ]:
#Tải script huấn luyện có giới hạn
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_train_separate_clipping_25(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_train_separate_clipping_25.py"

script = Path("/content/hr17_train_separate_clipping_25.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA HUẤN LUYỆN")

In [ ]:
#Chạy huấn luyện và đánh giá
import subprocess
from pathlib import Path

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_train_separate_clipping_25.py",
    "--run",
]

log_path = Path("/content/hr17_separate_clipping_25_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

In [ ]:
#Tải kết quả về máy
from google.colab import files
from pathlib import Path

archives = list(Path("/content/hr17_separate_clipping_25").glob("*.zip"))
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))

# **Phép thử dùng nguyên mã huấn luyện đã lưu, chạy hai lượt × 5 cập nhật, mỗi lượt bắt đầu từ checkpoint gốc.**

In [ ]:
#Tải script kiểm tra tính lặp lại
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_reproduce_separate5(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_reproduce_separate5.py"

script = Path("/content/hr17_reproduce_separate5.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA HUẤN LUYỆN")

In [ ]:
#Chạy hai lượt kiểm chứng
import subprocess
from pathlib import Path

source = Path(
    "/content/hr17_ppo_clipping_comparison/"
    "20261124_085613_561254.zip"
)
assert source.is_file(), "Không thấy ZIP nguồn trong phiên hiện tại."

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_reproduce_separate5.py",
    "--run",
    "--source-zip", str(source),
]

log_path = Path("/content/hr17_reproduce_separate5_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

In [ ]:
#Tải kết quả
from google.colab import files
from pathlib import Path

archives = list(Path("/content/hr17_reproduce_separate5").glob("*.zip"))
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))

# **Chạy 5 lần với checkpoint 2,26 giây và 5 lần với checkpoint gốc, cùng clip id01. Tổng cộng 10 lượt đánh giá, không huấn luyện thêm.**

In [ ]:
#Tải script đánh giá lặp lại
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_repeat_candidate_226(?: \(\d+\))?\.py", name
    )
]
assert len(matches) == 1, "Chọn file hr17_repeat_candidate_226.py"

script = Path("/content/hr17_repeat_candidate_226.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA CHẠY ĐÁNH GIÁ")

In [ ]:
#Chạy 10 lượt đánh giá
import subprocess
from pathlib import Path

source = Path(
    "/content/hr17_reproduce_separate5/"
    "20261124_101330_883796.zip"
)
assert source.is_file(), "Không thấy ZIP chứa checkpoint 2,26 giây."

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_repeat_candidate_226.py",
    "--run",
    "--source-zip", str(source),
]

log_path = Path("/content/hr17_repeat_candidate_226_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

In [ ]:
#Tải ZIP kết quả
from google.colab import files
from pathlib import Path

archives = list(Path("/content/hr17_repeat_candidate_226").glob("*.zip"))
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))

# **Phép thử gồm 20 lượt đánh giá trên id02, id03, ood01, ood02 và ood03, không huấn luyện thêm.**

In [ ]:
#Tải script đánh giá 5 chuyển động
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_validate_candidate226_5motions(?: \(\d+\))?\.py",
        name,
    )
]
assert len(matches) == 1, "Chọn file hr17_validate_candidate226_5motions.py"

script = Path("/content/hr17_validate_candidate226_5motions.py")
script.write_bytes(uploaded[matches[0]])
compile(script.read_text(encoding="utf-8"), str(script), "exec")

print("ĐÃ TẢI SCRIPT; CHƯA CHẠY ĐÁNH GIÁ")

In [ ]:
#Chạy đánh giá
import subprocess
from pathlib import Path

source = Path(
    "/content/hr17_repeat_candidate_226/"
    "20261124_102308_376890.zip"
)
assert source.is_file(), "Không thấy ZIP chứa checkpoint đã kiểm chứng."

command = [
    "/usr/bin/python3", "-u",
    "/content/hr17_validate_candidate226_5motions.py",
    "--run",
    "--source-zip", str(source),
]

log_path = Path("/content/hr17_candidate226_5motions_console.log")

with log_path.open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

In [ ]:
#Tải kết quả
from google.colab import files
from pathlib import Path

archives = list(
    Path("/content/hr17_validate_candidate226_5motions").glob("*.zip")
)
assert archives, "Chưa có ZIP kết quả."

latest = max(archives, key=lambda p: p.stat().st_mtime)
print("TẢI:", latest)
files.download(str(latest))